In [ ]:
from github import Github, Auth
import pandas as pd
from datetime import datetime, timedelta
import time
import os

# --- 1. Load AIDev blacklist ---
print("Loading AIDev blacklist...")
all_pr_df = pd.read_parquet("hf://datasets/hao-li/AIDev/all_pull_request.parquet")
all_repo_df = pd.read_parquet("hf://datasets/hao-li/AIDev/all_repository.parquet")

blacklisted_repo_names = set(all_repo_df['full_name'].unique())
blacklisted_user_ids = set(all_pr_df['user_id'].unique())

# --- 2. GitHub API ---
g = Github(auth=Auth.Token(os.getenv("GITHUB_TOKEN_1")))

# --- 3. Config ---
TARGET_COUNT = 100
MIN_STARS = 50
MIN_YEARS = 3
created_before = (datetime.now() - timedelta(days=MIN_YEARS * 365)).strftime('%Y-%m-%d')

# All major agents we want to filter
agent_keywords = [
    "copilot", "cursor", "devin", "codex",
    "claude", "gpt", "agent", "bot",
    "wizardcoder", "codewhisperer", "tabnine", "codeium"
]

# Agent-specific config/workflow files
agent_file_keywords = [
    ".copilot", ".cursor", ".devin", ".codex",
    ".claude", ".gpt", "copilot.yml", "cursor.yml",
    "devin.yml", "codex.yml", "claude.yml",
    "ai_workflow", "codegen", "wizardcoder", 
    "codewhisperer", "tabnine", "codeium"
]

# Agent PR detection: authors, branch names, co-authors
agent_pr_authors = ["devin-ai-integration[bot]"]
agent_pr_branch_prefixes = ["codex/", "copilot/", "cursor/"]
agent_pr_coauthors = ["Claude"]

print(f"Searching repos created before {created_before} with ≥{MIN_STARS} stars")
query = f"stars:>={MIN_STARS} created:<{created_before} is:public"
search = g.search_repositories(query=query, sort="stars", order="desc")

human_repos = []

# --- 4. Iterate safely ---
for repo in search:
    if len(human_repos) >= TARGET_COUNT:
        break

    # --- 4a. AIDev blacklist ---
    if repo.full_name in blacklisted_repo_names:
        continue
    if repo.owner.id in blacklisted_user_ids:
        continue

    desc = (repo.description or "").lower()
    name = repo.full_name.lower()
    owner_login = (repo.owner.login or "").lower()

    # # --- 4b. Name / description keywords ---
    # if any(k in desc for k in agent_keywords):
    #     continue
    # if any(k in name for k in agent_keywords):
    #     continue

    # --- 4c. Bot owner check ---
    if repo.owner.type == "Bot" or "bot" in owner_login:
        continue

    # # --- 4d. Check agent-specific files / workflows recursively ---
    skip_repo = False
    # try:
    #     contents = repo.get_contents("")
    #     while contents:
    #         file_content = contents.pop(0)
    #         path = file_content.path.lower()

    #         if any(k in path for k in agent_file_keywords):
    #             skip_repo = True
    #             break

    #         if file_content.type == "dir":
    #             try:
    #                 contents.extend(repo.get_contents(file_content.path))
    #             except:
    #                 continue
    # except Exception:
    #     pass  # ignore repos where listing fails

    # if skip_repo:
    #     continue

    # --- 4e. Check branch names for any agent keywords ---
    try:
        for branch in repo.get_branches():
            branch_name = branch.name.lower()
            if any(k in branch_name for k in agent_keywords):
                skip_repo = True
                break
    except Exception:
        pass

    if skip_repo:
        continue

    # --- 4f. Check PRs for agent activity ---
    try:
        for pr in repo.get_pulls(state="all"):
            # Author filter
            if pr.user.login in agent_pr_authors:
                skip_repo = True
                break

            # Branch name filter
            if any(pr.head.ref.lower().startswith(prefix) for prefix in agent_pr_branch_prefixes):
                skip_repo = True
                break

            # Check co-authors in commit messages
            pr_commits = pr.get_commits()
            for commit in pr_commits:
                commit_msg = commit.commit.message
                if any(f"Co-Authored-By: {author}" in commit_msg for author in agent_pr_coauthors):
                    skip_repo = True
                    break
            if skip_repo:
                break
    except Exception:
        pass  # ignore repos where PR listing fails

    if skip_repo:
        continue

    # --- 4g. Passed all filters ---
    human_repos.append({
        "full_name": repo.full_name,
        "url": repo.html_url,
        "created_at": repo.created_at.date()
    })

    print(f"[{len(human_repos)}/{TARGET_COUNT}] {repo.full_name}")
    time.sleep(1)

# --- 5. Save ---
with open("true_human_baseline.txt", "w", encoding="utf-8") as f:
    for r in human_repos:
        f.write(
            f"{r['full_name']} | {r['url']} | "
            f"0 agent PRs | first agent PR: 2025-06-02 | "
            f"repo creation: {r['created_at']}\n"
        )

print("Done.")

Loading AIDev blacklist...


In [5]:
import pandas as pd
import re

# Path to your text file
file_path = r"G:\CMPUT660Project\filtered_repos_3year50pr.txt"

# Read the file
with open(file_path, "r", encoding="utf-8") as f:
    text = f.read()

# Extract all "first agent PR: YYYY-MM-DD"
dates_str = re.findall(r"first agent PR:\s*(\d{4}-\d{2}-\d{2})", text)

# Convert to datetime and make a Series
dates = pd.Series(pd.to_datetime(dates_str))

# Compute median
median_date = dates.median()
print("Median first agent PR date:", median_date.date())

Median first agent PR date: 2025-06-02
